# Prepare training set

In [1]:
!ls ../data/species

Brachionus_asplanchnoidis  Drosophila_willistoni      Plasmodium_falciparum
Cloeon_dipterum		   Mnemiopsis_leidyi	      Plasmodium_vivax
Daphnia_magna		   Mytilus_galloprovincialis
Drosophila_melanogaster    Nematostella_vectensis


## Clean assemblies

In [2]:
tr_sp=!ls ../data/species
#tr_sp=["Mytilus_galloprovincialis"]
for sp in tr_sp:

    ref_assembly=!realpath ../data/species/$sp/raw_*.fna
    ref_assembly=ref_assembly[0]
    ref_assembly_name=ref_assembly.split("/")[-1].replace("raw_", "")
    result_file=f"../data/species/{sp}/CLEAN_{ref_assembly_name}"
    print(result_file)
    
    !bash ../scripts/clean_ref.sh $ref_assembly > $result_file

../data/species/Brachionus_asplanchnoidis/CLEAN_Basplanchnoidis_gn.fna
../data/species/Cloeon_dipterum/CLEAN_Cdipterum_gn.fna
../data/species/Daphnia_magna/CLEAN_Dmagna_gn.fna
../data/species/Drosophila_melanogaster/CLEAN_Dmelanogaster_gn.fna
../data/species/Drosophila_willistoni/CLEAN_Dwillistoni_gn.fna
../data/species/Mnemiopsis_leidyi/CLEAN_Mleidyi_gn.fna
../data/species/Mytilus_galloprovincialis/CLEAN_Mgalloprovincialis_gn.fna
../data/species/Nematostella_vectensis/CLEAN_Nvectensis_gn.fna
../data/species/Plasmodium_falciparum/CLEAN_Pfalciparum_gn.fna
../data/species/Plasmodium_vivax/CLEAN_Pvivax_gn.fna


## Extract CDS from annotations

In [3]:
tr_sp=!ls ../data/species
#tr_sp=["Mytilus_galloprovincialis"]
for sp in tr_sp:
    ref_ann=!realpath ../data/species/$sp/raw_*.gff
    ref_ann=ref_ann[0]
    ref_ann_name=ref_ann.split("/")[-1].replace("raw_", "")
    result_file=f"../data/species/{sp}/CDS_{ref_ann_name}"
    print(result_file)

    !bash ../scripts/get_CDS.sh $ref_ann > $result_file

../data/species/Brachionus_asplanchnoidis/CDS_Basplanchnoidis_ann.gff
../data/species/Cloeon_dipterum/CDS_Cdipterum_ann.gff
../data/species/Daphnia_magna/CDS_Dmagna_ann.gff
../data/species/Drosophila_melanogaster/CDS_Dmelanogaster_ann.gff
../data/species/Drosophila_willistoni/CDS_Dwillistoni_ann.gff
../data/species/Mnemiopsis_leidyi/CDS_Mleidyi_ann.gff
../data/species/Mytilus_galloprovincialis/CDS_Mgalloprovincialis_ann.gff
../data/species/Nematostella_vectensis/CDS_Nvectensis_ann.gff
../data/species/Plasmodium_falciparum/CDS_Pfalciparum_ann.gff
../data/species/Plasmodium_vivax/CDS_Pvivax_ann.gff


## Sample n genes from CDS annotations

In [4]:
tr_sp=!ls ../data/species
n=1000
#tr_sp=["Mytilus_galloprovincialis"]
for sp in tr_sp:
    CDS_ann=!realpath ../data/species/$sp/CDS*.gff
    CDS_ann=CDS_ann[0]
    CDS_ann_name=CDS_ann.split("/")[-1]
    result_file=f"../data/species/{sp}/sample{n}_{CDS_ann_name}"
    print(result_file)

    !bash ../scripts/sample_CDS.sh $CDS_ann $n > $result_file

../data/species/Brachionus_asplanchnoidis/sample1000_CDS_Basplanchnoidis_ann.gff
../data/species/Cloeon_dipterum/sample1000_CDS_Cdipterum_ann.gff
../data/species/Daphnia_magna/sample1000_CDS_Dmagna_ann.gff
../data/species/Drosophila_melanogaster/sample1000_CDS_Dmelanogaster_ann.gff
../data/species/Drosophila_willistoni/sample1000_CDS_Dwillistoni_ann.gff
../data/species/Mnemiopsis_leidyi/sample1000_CDS_Mleidyi_ann.gff
../data/species/Mytilus_galloprovincialis/sample1000_CDS_Mgalloprovincialis_ann.gff
../data/species/Nematostella_vectensis/sample1000_CDS_Nvectensis_ann.gff
../data/species/Plasmodium_falciparum/sample1000_CDS_Pfalciparum_ann.gff
../data/species/Plasmodium_vivax/sample1000_CDS_Pvivax_ann.gff


# Training commands

In [ ]:
tr_sp=!ls ../data/species
n=1000
#tr_sp=["Mytilus_galloprovincialis"]
!mkdir -p ../job_commands
with open("../job_commands/SAR_training.txt", "w") as out:
    
    #create trained parameters folder command
    parameters_folder="../results/trainedParams"
    print(f"mkdir -p {parameters_folder}", file=out)
    for sp in tr_sp:
        
        #get paths for cleaned data
        sampleN_CDS=!realpath ../data/species/$sp/sample$n*.gff
        sampleN_CDS=sampleN_CDS[0]
    
        clean_ref=!realpath ../data/species/$sp/CLEAN*.fna
        clean_ref=clean_ref[0]

        #set folder for results
        result_sp=f"../results/{sp}/"

        #training command using singularity container
        geneidtrainer_command=f"singularity run \
                            ../data/geneidtrainerdocker.sif \
                            /scripts_geneid/geneidTRAINer4docker.pl \
                            -species {sp} \
                            -gff {sampleN_CDS} \
                            -fastas {clean_ref} \
                            -results {result_sp} \
                            -reduced no"
        geneidtrainer_command=geneidtrainer_command.replace("                             ", " ")
        print(geneidtrainer_command)
        print(geneidtrainer_command, file=out)
        #!$geneidtrainer_command

        #copy parameter files for clear access(+git)
        #move them to folder that will be in git, and link them to species specific
        mvParam_command=f"mv {result_sp}*.optimized.param {parameters_folder}/"
        lnParam_command=f"ln -vs ../{parameters_folder}/{sp}.geneid.optimized.param {result_sp}"
        print(mvParam_command)
        print(mvParam_command, file=out)
        print(lnParam_command)
        print(lnParam_command, file=out)

#Execute generated commands

singularity run ../data/geneidtrainerdocker.sif /scripts_geneid/geneidTRAINer4docker.pl -species Brachionus_asplanchnoidis -gff /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/data/species/Brachionus_asplanchnoidis/sample1000_CDS_Basplanchnoidis_ann.gff -fastas /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/data/species/Brachionus_asplanchnoidis/CLEAN_Basplanchnoidis_gn.fna -results ../results/Brachionus_asplanchnoidis/ -reduced no
mv ../results/Brachionus_asplanchnoidis/*.optimized.param ../results/trainedParams/
ln -vs ../../results/trainedParams/Brachionus_asplanchnoidis.geneid.optimized.param ../results/Brachionus_asplanchnoidis/
singularity run ../data/geneidtrainerdocker.sif /scripts_geneid/geneidTRAINer4docker.pl -species Cloeon_dipterum -gff /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/data/species/Cloeon_dipterum/sample1000_CDS_Cdipterum_ann.gff -fastas /home/jj/Desktop/Data_science/CRG/TFM/projects/geneid-training/data/species/